## 0. __Imports__

In [1]:
from recipes import *
import itertools
import random
from typing import Dict
import time
import pandas as pd
import numpy as np
import joblib
from tqdm.auto import tqdm
from itertools import product

from sklearn.pipeline import Pipeline
from catboost import CatBoostRegressor, CatBoostClassifier

from sklearn.linear_model import (
	LinearRegression,
	LogisticRegression,
)

from sklearn.model_selection import (
	train_test_split, 
)

from sklearn.tree import (
	DecisionTreeRegressor, 
  	DecisionTreeClassifier,
)

from sklearn.ensemble import (
	RandomForestRegressor,
	RandomForestClassifier,
	VotingRegressor,
	VotingClassifier,
)

from sklearn.metrics import (
	root_mean_squared_error,
	accuracy_score,
	precision_score,
)

import sys
import asyncio
import aiohttp
from bs4 import BeautifulSoup


### 0.1. __Download files__

In [2]:
try:
	df = pd.read_csv('../datasets/epi_r.csv', encoding='utf-8')
except:
	file = 'https://drive.google.com/uc?export=download&id=1hzmxNBrY7-9mv5EpqAvhVUiJahfrcYUN'
	df = RecipeDataPreprocessor().clean_up_df(pd.read_csv(file, encoding="utf-8"))
	df.to_csv('../datasets/epi_r.csv', index=False)


## 1. __Forecast__

### 1.1. Data preparation

* Use [the dataset](https://drive.google.com/file/d/1hzmxNBrY7-9mv5EpqAvhVUiJahfrcYUN/view?usp=sharing) from Epicurious collected by HugoDarwood.

* Filter the columns: the less non-ingredient columns in your dataset the better. You will predict the rating or rating category using only the ingredients and nothing else.


In [3]:
blacklist = [
    "calories", "protein", "fat", "sodium", "#cakeweek", "#wasteless",
    "22-minute meals", "3-ingredient recipes", "30 days of groceries", "advance prep required", "breakfast", "brunch", "dinner",
    "lunch", "dessert", "appetizer", "snack", "snack week", "side", "quick","easy", "quick and healthy",
    "advance prep required", "kid-friendly", "vegan", "vegetarian", "pescatarian", "low cal", "low fat",
    "low sugar", "low","no sugar", "low carb", "low cholesterol", "low sodium", "fat free", "pot pie",
    "sugar conscious", "healthy", "high fiber", "dairy free", "peanut free", "tree nut free", "soy free",
    "kosher", "kosher for passover", "wheat","gluten-free", "no sugar added", "non-alcoholic", "alcoholic", "organic",
    "tested","improved", "no-cook", "freeze","chill", "freezer food", "raw", "grill", "grill","barbecue",
    "broil", "roast", "bake", "pan-fry", "deep-fry", "fry", "simmer","fruit","braise",
    "steam", "stir-fry", "slow cooker", "pressure cooker", "smoker", "microwave", "food processor",
    "blender", "potato salad", "double boiler", "ice cream machine", "juicer", "mandoline", "mortar and pestle",
    "candy thermometer", "cookbook critic", "paleo", "cook like a diner", "house cocktail", "epi loves the microwave", "epi + ushg",
    "sandwich theory", "friendsgiving", "family reunion", "anniversary", "birthday", "wedding", "engagement party",
    "party", "picnic", "buffet", "potluck", "tailgating", "cocktail party", "super bowl",
    "thanksgiving", "christmas", "christmas eve", "new year's eve", "new year's day", "meatball", "meatloaf",
    "shower", "ramekin", "valentine's day", "st. patrick's day", "halloween", "hanukkah", "rosh hashanah","yom kippur",
    "purim", "sukkot", "passover", "shavuot", "diwali", "kwanzaa", "ramadan",
    "easter", "fourth of july", "bastille day", "mardi gras", "lunar new year", "oscars", "father's day",
    "winter", "flaming hot summer", "back to school", "backyard bbq", "entertaining", "casserole","gratin", "pasta maker",
    "pressure cooker", "sandwich","sourdough", "d.c.", "lasagna", "hot drink",
    "house","garden", "quiche", "frittata", "fritter", "salad",
    "salad dressing", "pizza", "cupcake", "cake", "cookie", "cookies", "brownie",
    "muffin", "biscuit", "waffle", "pancake", "taco", "burrito", "omelet","stew","soup","stew",
    "smoothie", "ice cream", "sorbet", "crêpe", "cobbler","crumble", "custard",
    "candy", "marshmallow", "edible gift", "hors d'oeuvre", "condiment", "condiment","spread", "salsa",
    "sauce", "rub", "marinade", "marinate", "pickles", "parade",
    "suzanne goin", "dorie greenspan", "dip", "drink", "drinks", "aperitif", "digestif",
    "cocktail", "spirit", "connecticut", "dallas", "denver", "stuffing","dressing", "spring",
    "summer", "fall", "graduation", "pasadena", "boston", "beverly hills", "nancy silverton",
    "oktoberfest", "poker", "game night", "pie", "providence", "france", "italy", "spain",
    "germany", "switzerland", "australia", "england", "santa monica", "kansas city", "kentucky derby",
    "kidney friendly", "kitchen olympics", "idaho", "hummus", "minneapolis", "mixer", "no meat","no problem",
    "persian new year", "sauté", "self", "ireland", "israel", "japan", "egypt",
    "canada", "haiti", "jamaica", "mexico", "peru", "philippines", "bulgaria",
    "cuba", "dominican republic", "georgia", "guam", "arizona", "alabama", "alaska",
    "california", "colorado", "mother's day", "one-pot meal", "punch", "skewer", "florida",
    "hawaii", "illinois", "indiana", "iowa", "kansas", "kentucky", "louisiana",
    "maine", "maryland", "massachusetts", "labor day", "michigan", "minnesota", "mississippi",
    "missouri", "nebraska", "new hampshire", "new jersey", "new mexico", "new orleans", "cambridge",
    "new york", "bon appétit", "harpercollins", "gourmet", "frankenrecipe", "frozen dessert", "game",
    "bon app��tit", "north carolina", "ohio", "weelicious", "chill", "oklahoma", "oregon",
    "pennsylvania", "rhode island", "south carolina", "tennessee", "texas", "utah", "vermont",
    "virginia", "washington", "west virginia", "wisconsin", "las vegas", "los angeles", "brooklyn",
    "atlanta", "columbus", "anthony bourdain", "houston", "emeril lagasse", "miami", "long beach",
    "san francisco", "seattle", "portland", "pittsburgh", "louisville", "windsor", "westwood",
    "lancaster", "healdsburg", "yonkers", "paris", "aspen", "costa mesa", "pacific palisades",
    "st. louis", "cookbooks", "leftovers", 'title', 'camping', 'chicago', 'hollywood', 'jalapeño',
	'seafood', 'sherry', 'soufflé', 'spritzer', 'triple sec', 'vermouth', 'yuca',
	'campari', 'chartreuse', 'grappa', 'jícama', 'kahlúa', 'marscarpone', 'rosé',
]

In [4]:
prep_pipeline = Pipeline([
    ('preprocessor', RecipeDataPreprocessor(blacklist=blacklist)),
])

df_final = prep_pipeline.fit_transform(df)
df_final.to_csv("../datasets/filtered_dataset.csv", index=False)


In [5]:
df_final.head()

,rating,almond,amaretto,anchovy,anise,apple,apple juice,apricot,artichoke,arugula,...,masa,jam,jelly,milk,cream,phyllo,puff pastry dough,meringue,sweet potato,yam
0,2.500,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,4.375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3.750,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,3.125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 1.2. Regression

* Try different algorithms and their hyperparameters for rating prediction. Choose the best on cross-validation and find the score (RMSE) on the test subsample.

* Try different ensembles and their hyperparameters. Choose the best on cross-validation and find the score on the test subsample.

* Calculate the RMSE for a naive regressor that predicts the average rating.

In [6]:
X = df_final.drop(columns=['rating'])
y = df_final['rating']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, random_state=21)

In [8]:
param_grid_dtr = {
    'max_depth': [5, 10],
    'min_samples_split': [5, 10],
}

param_grid_rfr = {
    'n_estimators': [50],
    'max_depth': [10, 20],
    'min_samples_split': [5, 10],
}

param_grid_cbr = {
    'iterations': [150],
    'depth': [4, 6],
    'learning_rate': [0.05, 0.1],
}

models = {
    'DecisionTreeRegressor': (
        DecisionTreeRegressor(random_state=21),
        param_grid_dtr,
    ),
    'RandomForestRegressor': (
        RandomForestRegressor(random_state=21, n_jobs=1),
        param_grid_rfr,
    ),
    'CatBoostRegressor': (
        CatBoostRegressor(
            loss_function='RMSE',
            random_seed=21,
            thread_count=1,
            verbose=False,
            allow_writing_files=False,
        ),
        param_grid_cbr,
    ),
}


In [9]:
reg_selector = ModelSelector(models)

In [10]:
reg_selector.fit(X_train, y_train)

Estimator: DecisionTreeRegressor:   0%|          | 0/20 [00:00<?, ?it/s]

DecisionTreeRegressor: Лучшие параметры: {'max_depth': 5, 'min_samples_split': 10}
rmse на кросс-валидации: 1.3013


Estimator: RandomForestRegressor:   0%|          | 0/20 [00:00<?, ?it/s]

RandomForestRegressor: Лучшие параметры: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 50}
rmse на кросс-валидации: 1.2869


Estimator: CatBoostRegressor:   0%|          | 0/20 [00:00<?, ?it/s]

CatBoostRegressor: Лучшие параметры: {'depth': 6, 'iterations': 150, 'learning_rate': 0.1}
rmse на кросс-валидации: 1.2737

Лучшая модель: CatBoostRegressor
rmse = 1.2737


In [11]:
reg_selector.final_score(X_test, y_test)

Test rmse = 1.2626


#### Ensembles

In [12]:
param_grid_voting = {
    'weights': [[2, 3, 4]],
}

ensembels = {
    'voting': (
        VotingRegressor(
            list(tuple(reg_selector.best_models.items())),
            n_jobs=1,
        ),
        param_grid_voting,
    ),
}


In [13]:
ensembel_selector = ModelSelector(ensembels)

In [14]:
ensembel_selector.fit(X_train, y_train)

Estimator: voting:   0%|          | 0/5 [00:00<?, ?it/s]

voting: Лучшие параметры: {'weights': [2, 3, 4]}
rmse на кросс-валидации: 1.2799

Лучшая модель: voting
rmse = 1.2799


In [15]:
ensembel_selector.final_score(X_test, y_test)

Test rmse = 1.2707


#### Naive model

In [16]:
rmse_naive = root_mean_squared_error(y_test, np.full_like(y_test, fill_value=y_train.mean()))
print(f"RMSE = {rmse_naive}")

RMSE = 1.328147349527116


### 1.3. Classification

* Binarize the target column by rounding the ratings to the closest integer. This will be your classes.

* Try different algorithms and their hyperparameters for class prediction. Choose the best on cross-validation and find the score (accuracy) on the test subsample.

* Compare the metrics using accuracy. Calculate the accuracy of a naive classificator that predicts the most common class.

* Binarize the target column again by converting the integers to classes `‘bad’ (0, 1)`, `‘so-so’ (2, 3)`, `‘great’ (4, 5)`.

* Try different algorithms and their hyperparameters for class prediction. Choose the best on cross-validation and find the score on the test subsample.

* Compare the metrics using accuracy. Calculate the accuracy of a naive classificator that predicts the most common class.

* What is worse: to predict a bad rating which is good in real life, or to predict a good rating which is bad in real life? Replace accuracy with the appropriate metric.

* Try different algorithms and their hyperparameters for class prediction with the new metric. Choose the best and find the score on the test subsample.

* Try different ensembles and their hyperparameters. Choose the best and find the score on the test subsample.


#### Prepare dataset

In [17]:
df_classification = df_final.copy()
df_classification['rating'] =(df_final['rating'] + 0.5).astype(int)

In [18]:
df_classification.head()
 

,rating,almond,amaretto,anchovy,anise,apple,apple juice,apricot,artichoke,arugula,...,masa,jam,jelly,milk,cream,phyllo,puff pastry dough,meringue,sweet potato,yam
0,3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
y = df_classification['rating']
X = df_classification.drop(columns='rating')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)


In [20]:
y.value_counts()

rating
4    12118
5     2383
3     1838
0     1659
1      145
2      108
Name: count, dtype: int64

In [21]:
param_grid_dtc = {
    'criterion': ['gini'],
    'max_depth': [5, 10, None],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [1, 4],
}

param_grid_rfc = {
    'n_estimators': [100],
    'max_depth': [10, None],
    'min_samples_leaf': [1, 4],
    'bootstrap': [True],
}

param_grid_cbc = {
    'iterations': [150],
    'depth': [4, 6],
    'learning_rate': [0.05, 0.1],
}

models_classification = {
    'DTC': (
        DecisionTreeClassifier(class_weight='balanced', random_state=21),
        param_grid_dtc,
    ),
    'RFC': (
        RandomForestClassifier(class_weight='balanced', random_state=21, n_jobs=1),
        param_grid_rfc,
    ),
    'CatBoostClassifier': (
        CatBoostClassifier(
            loss_function='MultiClass',
            auto_class_weights='Balanced',
            random_seed=21,
            thread_count=1,
            verbose=False,
            allow_writing_files=False,
        ),
        param_grid_cbc,
    ),
}


In [22]:
classifier_selector = ModelSelector(models_classification, scorer='accuracy')

In [23]:
classifier_selector.fit(X_train, y_train)

Estimator: DTC:   0%|          | 0/60 [00:00<?, ?it/s]

DTC: Лучшие параметры: {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5}
accuracy на кросс-валидации: 0.3787


Estimator: RFC:   0%|          | 0/20 [00:00<?, ?it/s]

RFC: Лучшие параметры: {'bootstrap': True, 'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 100}
accuracy на кросс-валидации: 0.5939


Estimator: CatBoostClassifier:   0%|          | 0/20 [00:00<?, ?it/s]

CatBoostClassifier: Лучшие параметры: {'depth': 6, 'iterations': 150, 'learning_rate': 0.1}
accuracy на кросс-валидации: 0.2895

Лучшая модель: RFC
accuracy = 0.5939


In [24]:
classifier_selector.final_score(X_test, y_test)

Test accuracy = 0.5971


#### Уменьшение количества классов 

In [25]:
df_classification_reduced = df_classification.copy()

In [26]:
def map_rating_to_class(r):
    if r <= 1:
        return 'bad'
    elif r <= 3:
        return 'so-so'
    else:
        return 'great'

df_classification_reduced['rating'] = df_classification_reduced['rating'].apply(map_rating_to_class)


In [27]:
df_classification_reduced['rating'].value_counts()

rating
great    14501
so-so     1946
bad       1804
Name: count, dtype: int64

In [28]:
y = df_classification_reduced['rating']
X = df_classification_reduced.drop(columns='rating')

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)


In [30]:
classifier_selector_reduced = ModelSelector(models_classification, scorer='accuracy')

In [31]:
classifier_selector_reduced.fit(X_train, y_train)

Estimator: DTC:   0%|          | 0/60 [00:00<?, ?it/s]

DTC: Лучшие параметры: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 5}
accuracy на кросс-валидации: 0.5629


Estimator: RFC:   0%|          | 0/20 [00:00<?, ?it/s]

RFC: Лучшие параметры: {'bootstrap': True, 'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 100}
accuracy на кросс-валидации: 0.7420


Estimator: CatBoostClassifier:   0%|          | 0/20 [00:00<?, ?it/s]

CatBoostClassifier: Лучшие параметры: {'depth': 6, 'iterations': 150, 'learning_rate': 0.1}
accuracy на кросс-валидации: 0.4663

Лучшая модель: RFC
accuracy = 0.7420


In [32]:
classifier_selector_reduced.final_score(X_test, y_test)

Test accuracy = 0.7472


#### Naive model

In [33]:
most_frequent_class = y_test.value_counts().idxmax()
y_pred_naive = [most_frequent_class] * len(y_test)

accuracy_naive = accuracy_score(y_test, y_pred_naive)
accuracy_naive

0.7945768282662284

What is worse: to predict a bad rating which is good in real life, or to predict a good rating which is bad in real life? Replace accuracy with the appropriate metric.

* лучше предсказать хорошее блюдо в класс плохих

In [34]:
classifier_selector_reduced_precision = ModelSelector(models_classification, scorer='precision', pos_label='great')

In [35]:
classifier_selector_reduced_precision.fit(X_train, y_train)

Estimator: DTC:   0%|          | 0/60 [00:00<?, ?it/s]

DTC: Лучшие параметры: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 5}
precision на кросс-валидации: 0.8400


Estimator: RFC:   0%|          | 0/20 [00:00<?, ?it/s]

RFC: Лучшие параметры: {'bootstrap': True, 'max_depth': 10, 'min_samples_leaf': 4, 'n_estimators': 100}
precision на кросс-валидации: 0.8496


Estimator: CatBoostClassifier:   0%|          | 0/20 [00:00<?, ?it/s]

CatBoostClassifier: Лучшие параметры: {'depth': 6, 'iterations': 150, 'learning_rate': 0.05}
precision на кросс-валидации: 0.8549

Лучшая модель: CatBoostClassifier
precision = 0.8549


In [36]:
classifier_selector_reduced_precision.final_score(X_test, y_test)

Test precision = 0.8513


#### Ensembles

In [37]:
param_grid_voting_cl = {
    'voting': ['soft'],
    'weights': [[1, 2, 2]],
}

ensembels_cl = {
    'voting': (
        VotingClassifier(
            list(tuple(classifier_selector_reduced_precision.best_models.items())),
            n_jobs=1,
        ),
        param_grid_voting_cl,
    ),
}


In [38]:
ensembel_selector_cl = ModelSelector(ensembels_cl, scorer='precision', pos_label='great')

In [39]:
ensembel_selector_cl.fit(X_train, y_train)

Estimator: voting:   0%|          | 0/5 [00:00<?, ?it/s]

voting: Лучшие параметры: {'voting': 'soft', 'weights': [1, 2, 2]}
precision на кросс-валидации: 0.8498

Лучшая модель: voting
precision = 0.8498


In [40]:
ensembel_selector_cl.final_score(X_test, y_test)

Test precision = 0.8502


### 1.4. Decision

* Decide what is better to use: the regression model or the classification.

Save the best model. You will use it in the program.

In [41]:
best_model = classifier_selector_reduced_precision.final_model

In [42]:
joblib.dump(best_model, 'best_model.pkl')


['best_model.pkl']

## 2. Nutrition Facts


* Collect all the nutrition facts for the ingredients from your prepared and filtered dataset (only ingredient columns) into a dataframe. Use [the following API](https://fdc.nal.usda.gov/api-guide.html) for that.

* Transform all the values into % of the daily value. Keep only nutrients that exist in [this](https://drive.google.com/file/d/1jn0t5tU_RgOpq4wcO-uS4D0_NAP6MwHz/view?usp=sharing) and [that](https://drive.google.com/file/d/1bmdZGB0QwND2BD3XlC1JswL7AdnTJHLT/view?usp=sharing) table.

* Save the transformed dataframe into a CSV file that you will use in your main program


In [43]:
nutr = NutritionFacts(X.columns.to_list(), 'batwomal_API_KEY')

In [44]:
nutr.nutrients_facts.head(10)

,almond,amaretto,anchovy,anise,apple,apple juice,apricot,artichoke,arugula,asian pear,...,masa,jam,jelly,milk,cream,phyllo,puff pastry dough,meringue,sweet potato,yam
Nutrient,,,,,,,,,,,,,,,,,,,,,
Fat,28.202247,5.617978,2.719101,8.932584,0.365169,0.000000,0.000000,3.747191,0.393258,0.129213,...,1.533708,0.000000,0.000000,1.191011,10.730337,3.370787,16.067416,0.000000,0.719101,0.000000
Saturated fat,0.000000,2.775000,1.066667,0.488333,0.000000,0.000000,0.000000,1.391667,0.000000,0.010000,...,0.000000,0.000000,0.000000,1.058333,8.500000,1.225000,9.916667,0.000000,0.266667,0.000000
Cholesterol,0.000000,0.000000,20.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,2.666667,19.666667,0.000000,0.000000,0.000000,4.666667,0.000000
Total carbohydrates,1.270588,2.611765,0.000000,3.921569,1.121569,0.980392,1.058824,0.261176,0.276078,0.831373,...,3.992157,4.705882,5.098039,0.398431,0.287059,4.125490,2.611765,5.231373,1.286275,2.196078
Sodium,0.038870,2.913043,4.521739,0.695652,0.000000,0.521739,0.260870,13.043478,1.217391,0.000000,...,0.000000,0.000000,3.260870,2.391304,3.130435,21.000000,16.565217,2.086957,7.521739,0.391304
Dietary Fiber,7.242188,0.000000,0.000000,11.406250,2.500000,0.000000,0.000000,1.328125,1.093750,2.812500,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.484375,3.750000,0.000000,1.484375,3.125000
Protein,17.466667,0.000000,13.600000,11.733333,0.000000,0.000000,0.000000,1.113333,1.880000,0.333333,...,4.846667,0.000000,0.000000,2.260000,1.973333,4.733333,0.000000,0.000000,3.633333,0.886667
Added sugars,0.000000,22.200000,0.000000,0.000000,6.933333,8.333333,9.000000,1.113333,1.406667,4.700000,...,0.000000,36.666667,43.333333,3.386667,2.446667,0.120000,0.000000,31.733333,2.780000,0.446667
Vitamin A,0.000000,0.000000,1.666667,10.366667,2.166667,0.000000,12.266667,55.666667,13.333333,0.000000,...,12.133333,0.000000,0.000000,7.066667,15.066667,0.000000,0.000000,0.000000,176.333333,4.433333


In [45]:
nutr.print_facts_for_ingredients(['gin', 'rum'])

Gin
Copper - 2.33% of Daily Value
Iron - 0.22% of Daily Value
Manganese - 0.78% of Daily Value
Niacin - 0.08% of Daily Value
Phosphorus - 0.32% of Daily Value
Potassium - 0.04% of Daily Value
Riboflavin - 0.31% of Daily Value
Sodium - 0.04% of Daily Value
Thiamin - 0.50% of Daily Value
Vitamin B6 - 0.06% of Daily Value
Zinc - 0.36% of Daily Value
Rum
Copper - 5.56% of Daily Value
Iron - 0.67% of Daily Value
Manganese - 0.78% of Daily Value
Phosphorus - 0.40% of Daily Value
Potassium - 0.04% of Daily Value
Sodium - 0.04% of Daily Value
Thiamin - 0.67% of Daily Value
Zinc - 0.64% of Daily Value


In [46]:
nutr.print_facts_for_ingredients(['gin', 'rum', 'beer'], total=True)

Added sugars - 6.06% of Daily Value
Copper - 7.89% of Daily Value
Iron - 0.89% of Daily Value
Manganese - 1.57% of Daily Value
Niacin - 0.08% of Daily Value
Phosphorus - 0.72% of Daily Value
Potassium - 0.09% of Daily Value
Riboflavin - 0.31% of Daily Value
Sodium - 0.43% of Daily Value
Thiamin - 1.17% of Daily Value
Total carbohydrates - 0.75% of Daily Value
Vitamin B6 - 0.06% of Daily Value
Zinc - 1.00% of Daily Value


## 3. __Similar Recipes__

* For each recipe from the dataset, collect the URL from epicurious.com with its details (if there is no URL for that recipe, skip it).

* Save the new dataframe to a CSV file that you will use in your main program.


In [47]:
async def fetch_response(
    session, 
    url: str, 
    headers: dict,
    params: dict={},
) -> str:
    """Асинхронно получает ответ от указанного URL.
    
    Args:
        session: Объект клиентской сессии aiohttp.
        url (str): URL для запроса.
        params (dict): Параметры запроса.
        headers (dict): Заголовки запроса.
    
    Returns:
        str: Текст ответа сервера.
    
    Raises:
        ValueError: Если статус ответа не равен 200.
    """
    async with session.get(url, params=params, headers=headers) as response:
        if response.status != 200:
            raise ValueError(f'Request error! Code: {response.status}')
        return await response.text()


In [48]:
def save_result_df(filename, df):
    if isinstance(df, pd.DataFrame):
        df.to_csv(filename, encoding='utf-8', index=False)
        sys.stdout.write('Save done!')
        return
    sys.stdout.write('There is nothing to save...')


In [49]:
def get_soup(response):
    return BeautifulSoup(response, 'lxml').find_all(
        'div',
        class_='ContentWrapper-bHDaQH fHXMRX',
    )

def get_recipe_name(div):
    return div.find(
        'h2',
        class_='BaseWrap-sc-gjQpdd BaseText-ewhhUZ Hed-hfYgrM iUEiRd euMCMz hqUziD'
    ).text.strip().casefold()

def get_ingredients_for_recipe(response):
    soup = BeautifulSoup(response, 'lxml').find_all(
        'div',
        class_='BaseWrap-sc-gjQpdd BaseText-ewhhUZ Description-cSrMCf iUEiRd bGCtOd fsKnGI'
    )
    ingredients = [ingredient.text for ingredient in soup]
    return f"{'; '.join(ingredients)}"

def get_recipe_url(base_url, div):
    return base_url + div.find(
        'a', 
        class_='BaseWrap-sc-gjQpdd BaseText-ewhhUZ BaseLink-eNWuiM Link-ehXskl iUEiRd bjMGUM kxxKIa'
    ).get('href')

def get_epicurious_rating(div):
    v = div.find(
        'p', 
        class_='BaseWrap-sc-gjQpdd BaseText-ewhhUZ RatingRating-btVmKd iUEiRd jjBDuu fOfiAf'
    )
    return (round(float(v.text), 2) if v else None)


In [50]:
async def search_recipe(
    session,
    semaphore: asyncio.Semaphore,
    **kwargs: dict
) -> dict:
    """Асинхронно ищет рецепт по заданным параметрам.

    Args:
        session: Объект клиентской сессии aiohttp.
        semaphore (asyncio.Semaphore): Семафор для ограничения количества одновременных запросов.
        **kwargs: Аргументы для поиска:
            - base_url (str): Базовый URL для поиска
            - params (dict): Параметры запроса
            - headers (dict): Заголовки запроса
            - recipe (str): Название искомого рецепта
            - result_dict (dict): Словарь для сохранения результатов
    
    Returns:
        dict: Словарь с результатами поиска, содержащий:
            - title (str): Название рецепта
            - epicurious_url (str): URL рецепта на сайте
            - epicurious_rating (float | None): Рейтинг рецепта (если есть)
    
    Note:
        Функция изменяет переданный словарь result_dict в kwargs.
    """
    async with semaphore:
        while True:
            try:
                response = await fetch_response(
                    session,
                    kwargs['base_url'] + '/search',
                    kwargs['headers'],
                    kwargs['params'],
                )
                soup = get_soup(response)
                for div in soup:
                    recipe_name = get_recipe_name(div)
                    if recipe_name == kwargs['recipe'].casefold():
                        recipe_url = get_recipe_url(kwargs['base_url'], div)
                        kwargs['result_dict'] = {
                            'recipe': kwargs['recipe'],
                            'epicurious_rating': get_epicurious_rating(div),
                            'epicurious_url': recipe_url,
                            'ingredients': get_ingredients_for_recipe(await fetch_response(session,
                                                                                           recipe_url,
                                                                                           kwargs['headers'])),
                        }
                        return kwargs['result_dict']
                
                kwargs['params']['page'] += 1
            
            except ValueError:
                break


In [51]:
async def collect_similar_recipes(
    df: pd.DataFrame,
    save_filename: str = None,
    num_responses: int = 15,
) -> list[dict]:
    """Асинхронно собирает информацию о рецептах с сайта Epicurious.
    
    Args:
        df (pd.DataFrame): DataFrame с рецептами, должен содержать столбец 'title'.
        num_responses (int): Максимальное количество одновременных запросов.
    
    Returns:
        list[dict]: Список словарей с информацией о найденных рецептах, каждый содержит:
            - title (str): Название рецепта
            - epicurious_url (str | None): URL рецепта на сайте (если найден)
            - epicurious_rating (float | None): Рейтинг рецепта (если есть)
    
    Note:
        Использует tqdm для отображения прогресса выполнения задач.
    """
    semaphore = asyncio.Semaphore(num_responses)
    df = df.copy().drop_duplicates()
    
    async with aiohttp.ClientSession() as session:
        tasks = []
        for recipe in df['title']:
            recipe = recipe.strip()
            kwargs = {
                'recipe': recipe,
                'base_url': 'https://www.epicurious.com',
                'headers': {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                            'AppleWebKit/537.36 (KHTML, like Gecko) '
                            'Chrome/91.0.4472.124 Safari/537.36'},
                'params': {'q': f"'{recipe}'",
                           'page': 1,
                           'content': 'recipe',},
                'result_dict': {'title': recipe,
                                'epicurious_rating': None,
                                'epicurious_url': None,
                                'ingredients': None,},
            }
            tasks.append(search_recipe(session, semaphore, **kwargs))
        
        available_recipes_list = []
        for task in tqdm(asyncio.as_completed(tasks),
                         total=len(tasks),
                         desc='📊 Loading Similar recipes',
                         colour="#00ff00",):
            task = await task

            if task:
                available_recipes_list.append(task)

    if save_filename:
        save_result_df(filename=save_filename, df=pd.DataFrame(available_recipes_list))


In [52]:
# await collect_similar_recipes(df.iloc[:, :], save_filename='similar_recipes.csv', num_responses=15)


In [53]:
recipese_df = pd.read_csv('../datasets/similar_recipes.csv').rename(columns={'recipe': 'title'})
recipese_df.head(10)

,title,epicurious_rating,epicurious_url,ingredients
0,Garlic Bread Puddings,3.6,https://www.epicurious.com/recipes/food/views/...,"2 cups milk; 8 garlic cloves, chopped; 2 large..."
1,Grilled Pizzas Provençale,4.0,https://www.epicurious.com/recipes/food/views/...,1 1/2 cups chopped seeded plum tomatoes (about...
2,Picadillo Tostadas,4.0,https://www.epicurious.com/recipes/food/views/...,"1 large onion, chopped fine; 2 garlic cloves, ..."
3,"Eggplant, Zucchini, Red Pepper, and Parmesan T...",4.5,https://www.epicurious.com/recipes/food/views/...,2 large onions (about 1 pound); 1 garlic clove...
4,Braised Birria,5.0,https://www.epicurious.com/recipes/food/views/...,"4 ounces ancho chiles (about 10), seeds remove..."
5,"Arugula, Yellow Tomato, and Nectarine Salsa",4.0,https://www.epicurious.com/recipes/food/views/...,1 1/2 cups 1/3-inch cubes pitted white nectari...
6,Frozen Strawberry-Banana Cake with Strawberry ...,4.0,https://www.epicurious.com/recipes/food/views/...,1 12-ounce pound cake; 1 cup strawberry preser...
7,Curried Pork Chops with Brandied Peach and Cra...,4.0,https://www.epicurious.com/recipes/food/views/...,"1 onion, diced; 1 1/2 tablespoons vegetable oi..."
8,Roasted Barley or Corn Tea,4.0,https://www.epicurious.com/recipes/food/views/...,1/2 cup roasted barley or 1 cup roasted corn; ...
9,Sauteed Escarole with Currants and Capers,4.0,https://www.epicurious.com/recipes/food/views/...,1 tablespoon pine nuts; 2 heads escarole (abou...


In [54]:
ingredients_list = ['apple', 'milk/cream', 'turkey']

In [55]:
similar_recipes = choose_similar_recipes(df,recipese_df,ingredients_list)
similar_recipes

,title,epicurious_rating,epicurious_url,ingredients
1325,Cider Mini-Doughnuts,3.4,https://www.epicurious.com/recipes/food/views/...,1 cup unfiltered apple cider; 3 1/2 cups all-p...
14238,Chicken with Sauteed Pears and Rosemary Sauce,4.0,https://www.epicurious.com/recipes/food/views/...,3 tablespoons plus 2 teaspoons vegetable oil; ...
10192,Herb-Roasted Turkey with Apple Cider Gravy,4.7,https://www.epicurious.com/recipes/food/views/...,8 quarts cold water; 2 cups coarse kosher salt...


In [56]:
print_similar_recipes(similar_recipes)


- Cider Mini-Doughnuts, rating: 3.4
URL: https://www.epicurious.com/recipes/food/views/cider-mini-doughnuts-240597

- Chicken with Sauteed Pears and Rosemary Sauce, rating: 4.0
URL: https://www.epicurious.com/recipes/food/views/chicken-with-sauteed-pears-and-rosemary-sauce-5845

- Herb-Roasted Turkey with Apple Cider Gravy, rating: 4.7
URL: https://www.epicurious.com/recipes/food/views/herb-roasted-turkey-with-apple-cider-gravy-108793
